# Predictive Modeling of Vehicle Fuel Efficiency (MPG)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = sns.load_dataset('mpg')

print("--- PREVIEW OF FIRST 5 ROWS ---")
display(df.head())

print("\n--- DATASET INFORMATION ---")
df.info()

print("\n--- STATISTICAL SUMMARY ---")
display(df.describe())


### Verify the specific count of missing values

In [ ]:
 
missing_hp = df['horsepower'].isnull().sum()
print(f"Missing values in horsepower before imputation: {missing_hp}")

median_hp = df['horsepower'].median()
df['horsepower'] = df['horsepower'].fillna(median_hp)

print(f"Missing values after imputation: {df['horsepower'].isnull().sum()}")

In [ ]:
plt.figure(figsize=(12, 5), facecolor='black')
ax = plt.gca()
ax.set_facecolor('black')

sns.histplot(df['mpg'], color='#00FFFF', bins=30, alpha=0.9, edgecolor='black', ax=ax, label='MPG Frequencies')

plt.title("UNIVARIATE ANALYSIS: VEHICLE FUEL EFFICIENCY (MPG) DISTRIBUTION", color='white', fontsize=14, pad=20)
plt.xlabel("Miles Per Gallon (MPG)", color='white', fontsize=12)
plt.ylabel("Frequency / Count", color='white', fontsize=12)
ax.tick_params(axis='both', colors='white', labelsize=10)

for spine in ax.spines.values():
    spine.set_color('#39FF14') # Neon Green frame
    spine.set_linewidth(2)

plt.legend(facecolor='black', labelcolor='white', edgecolor='#00FFFF')
plt.grid(True, linestyle=':', alpha=0.2, color='white')
plt.show()

In [ ]:
plt.figure(figsize=(12, 5), facecolor='black')
ax = plt.gca()
ax.set_facecolor('black')

plt.scatter(df['weight'], df['mpg'], color='#FF0000', alpha=0.5, label='Vehicle Data Points')

plt.title("BIVARIATE ANALYSIS: WEIGHT VS. FUEL EFFICIENCY", color='white', fontsize=14, pad=20)
plt.xlabel("Vehicle Weight (lbs)", color='white', fontsize=12)
plt.ylabel("Miles Per Gallon (MPG)", color='white', fontsize=12)
ax.tick_params(axis='both', colors='white', labelsize=10)

for spine in ax.spines.values():
    spine.set_color('#39FF14') # Neon Green frame
    spine.set_linewidth(2)

plt.legend(facecolor='black', labelcolor='white', edgecolor='#00FFFF')
plt.grid(True, linestyle=':', alpha=0.2, color='white')
plt.show()

In [ ]:
plt.figure(figsize=(12, 8), facecolor='black')
ax = plt.gca()
ax.set_facecolor('black')

numeric_df = df.select_dtypes(include=[np.number])
corr = numeric_df.corr()


heatmap = sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5,
                      ax=ax, cbar_kws={'label': 'Pearson Correlation Coefficient'})

cbar = heatmap.collections[0].colorbar
cbar.ax.yaxis.set_tick_params(color='white')
plt.setp(cbar.ax.yaxis.get_ticklabels(), color='white')
cbar.set_label('Pearson Correlation Coefficient', color='white')

for text in ax.texts:
    text.set_color("black" if abs(float(text.get_text())) > 0.4 else "white")

plt.title("MULTIVARIATE ANALYSIS: FEATURE CORRELATION MATRIX", color='white', fontsize=14, pad=20)
ax.tick_params(axis='both', colors='white', labelsize=10)

for spine in ax.spines.values():
    spine.set_color('#39FF14') # Neon Green frame
    spine.set_linewidth(2)

plt.show()

## 1.Linear Regression

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

features = ['cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model_year', 'origin']
X = df[features]
y = df['mpg']

X = pd.get_dummies(X, columns=['origin'], drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

y_pred = lr_model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("--- BASELINE LINEAR REGRESSION RESULTS ---")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"R-squared (R^2): {r2:.2f}")

## 2. Polynomial Regression

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

features = ['cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model_year', 'origin']
X = df[features]
y = df['mpg']
X = pd.get_dummies(X, columns=['origin'], drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


poly_model = Pipeline([
    ("poly_features", PolynomialFeatures(degree=2, include_bias=False)),
    ("linear_regression", LinearRegression())
])

poly_model.fit(X_train, y_train)

y_pred_poly = poly_model.predict(X_test)
mse_poly = mean_squared_error(y_test, y_pred_poly)
r2_poly = r2_score(y_test, y_pred_poly)

print("--- POLYNOMIAL REGRESSION RESULTS (Degree 2) ---")
print(f"Mean Squared Error (MSE): {mse_poly:.2f}")
print(f"R-squared (R^2): {r2_poly:.2f}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures

X_plot = df[['weight']].sort_values(by='weight')
y_plot = df.loc[X_plot.index, 'mpg']

lr = LinearRegression()
lr.fit(df[['weight']], df['mpg'])
y_stack_lr = lr.predict(X_plot)

poly = PolynomialFeatures(degree=2)
X_poly = poly.fit_transform(df[['weight']])
poly_lr = LinearRegression()
poly_lr.fit(X_poly, df['mpg'])
y_stack_poly = poly_lr.predict(poly.transform(X_plot))

plt.figure(figsize=(12, 7), facecolor='white')
plt.scatter(df['weight'], df['mpg'], color='gray', alpha=0.3, label='Actual Data (398 instances)')

plt.plot(X_plot, y_stack_lr, color='red', linewidth=3, label=f'Linear Baseline (R²: 0.84)')

plt.plot(X_plot, y_stack_poly, color='blue', linewidth=3, label=f'Polynomial Degree 2 (R²: 0.89)')

plt.title("Regression Comparison: Linear Baseline vs. Polynomial Curve", fontsize=14)
plt.xlabel("Vehicle Weight (lbs)", fontsize=12)
plt.ylabel("Miles Per Gallon (MPG)", fontsize=12)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()


## 3. Regularized Linear Models (Ridge & Lasso)

In [ ]:
from sklearn.linear_model import Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline


ridge_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('ridge', Ridge(alpha=1.0))
])
ridge_pipe.fit(X_train, y_train)
y_pred_ridge = ridge_pipe.predict(X_test)

lasso_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('lasso', Lasso(alpha=0.1))
])
lasso_pipe.fit(X_train, y_train)
y_pred_lasso = lasso_pipe.predict(X_test)

# Evaluation
print("--- REGULARIZATION RESULTS ---")
print(f"Ridge L2 MSE: {mean_squared_error(y_test, y_pred_ridge):.2f} | R2: {r2_score(y_test, y_pred_ridge):.2f}")
print(f"Lasso L1 MSE: {mean_squared_error(y_test, y_pred_lasso):.2f} | R2: {r2_score(y_test, y_pred_lasso):.2f}")



In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np


features = X.columns
coef_lr = lr_model.coef_
coef_ridge = ridge_pipe.named_steps['ridge'].coef_
coef_lasso = lasso_pipe.named_steps['lasso'].coef_

coef_df = pd.DataFrame({
    'Feature': features,
    'Linear': coef_lr,
    'Ridge': coef_ridge,
    'Lasso': coef_lasso
}).melt(id_vars='Feature', var_name='Model', value_name='Coefficient')

plt.figure(figsize=(12, 6))
sns.barplot(data=coef_df, x='Feature', y='Coefficient', hue='Model')
plt.title("Comparison of Model Coefficients: Linear vs. Ridge vs. Lasso", fontsize=14)
plt.axhline(0, color='black', linewidth=0.8)
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()


## 4.Support Vector Regressors (SVR)

In [ ]:
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score


svr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('svr', SVR(kernel='rbf', C=100, epsilon=0.1))
])

svr_pipe.fit(X_train, y_train)

y_pred_svr = svr_pipe.predict(X_test)
mse_svr = mean_squared_error(y_test, y_pred_svr)
r2_svr = r2_score(y_test, y_pred_svr)

print("--- SUPPORT VECTOR REGRESSOR (SVR) RESULTS ---")
print(f"Mean Squared Error (MSE): {mse_svr:.2f}")
print(f"R-squared (R^2): {r2_svr:.2f}")

## 5.Random Forest Ensemble

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score


rf_model = RandomForestRegressor(n_estimators=100, random_state=42)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
mse_rf = mean_squared_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print("--- RANDOM FOREST ENSEMBLE RESULTS ---")
print(f"Mean Squared Error (MSE): {mse_rf:.2f}")
print(f"R-squared (R^2): {r2_rf:.2f}")


## Feature Importance

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

importances = rf_model.feature_importances_
feature_names = X.columns
rf_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(
    data=rf_importance_df, 
    x='Importance', 
    y='Feature', 
    hue='Feature',    
    palette='viridis', 
    legend=False      
)

plt.title("Random Forest: Feature Importance for Predicting MPG", fontsize=14)
plt.xlabel("Relative Importance Score", fontsize=12)
plt.ylabel("Vehicle Attribute", fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression

estimators = {
    "Baseline Linear Regression": Pipeline([('scaler', StandardScaler()), ('model', LinearRegression())]),
    
    "Ridge Regression (L2)": ridge_pipe,
    "Lasso Regression (L1)": lasso_pipe,
    "Support Vector Regressor (SVR)": svr_pipe, 
    
    "Random Forest Ensemble": Pipeline([('scaler', StandardScaler()), ('model', rf_model)]),
    # -----------------------------------
    
    "Polynomial Regression (Deg 2)": Pipeline([
        ('scaler', StandardScaler()), 
        ('poly', PolynomialFeatures(degree=2)), 
        ('model', LinearRegression())
    ])
}

print("=== Starting Final 5-Fold Cross-Validation Tournament ===")
print(f"{'Model':<32} | {'True CV R² (Std)':<18} | {'True CV MSE (Std)':<18}")
print("-" * 75)

for name, pipeline in estimators.items():
    cv_results = cross_validate(
        pipeline, X_train, y_train, cv=10, 
        scoring=('neg_mean_squared_error', 'r2'), n_jobs=-1
    )
    
    mse_scores = -cv_results['test_neg_mean_squared_error']
    r2_scores = cv_results['test_r2']
    
    print(f"{name:<32} | {r2_scores.mean():.2f} (+/- {r2_scores.std():.2f}) | {mse_scores.mean():.2f} (+/- {mse_scores.std():.2f})")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error


X_train_72, X_test_72, y_train_72, y_test_72 = train_test_split(
    X, y, test_size=0.2, random_state=72
)

print("=== Single Split Evaluation (Random State = 72) ===")
print(f"{'Model':<32} | {'Holdout R²':<12} | {'Holdout MSE':<12}")
print("-" * 62)

for name, pipeline in estimators.items():
    pipeline.fit(X_train_72, y_train_72)
    
    y_pred = pipeline.predict(X_test_72)
    
    holdout_r2 = r2_score(y_test_72, y_pred)
    holdout_mse = mean_squared_error(y_test_72, y_pred)
    
    print(f"{name:<32} | {holdout_r2:.4f}       | {holdout_mse:.2f}")